In [1]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath("../../../"))

from bmad.core.workflows.challenger_solver.challenger_solver import (
    challenger_solver_loop,
    Critique
)

# Mock Solver: Generates a sort function
def mock_solver(problem, critique=None):
    if critique is None:
        # Initial solution (buggy)
        return "def sort_list(lst): return sorted(lst)"
    else:
        # Fixed solution based on critique
        if "empty list" in critique.feedback.lower():
            return "def sort_list(lst): return sorted(lst) if lst else []"
        return "def sort_list(lst): return sorted(lst)"

# Mock Challenger: Critiques the solution
iteration_count = 0
def mock_challenger(solution):
    global iteration_count
    iteration_count += 1
    
    if "if lst else" in solution:
        # Good solution
        return Critique(
            score=0.9,
            feedback="Solution handles empty list correctly.",
            issues=[],
            suggestions=[]
        )
    else:
        # Needs improvement
        return Critique(
            score=0.5,
            feedback="What about empty list?",
            issues=["Does not handle empty list explicitly"],
            suggestions=["Add check for empty list"]
        )

# Run Challenger/Solver Loop
print("Running Challenger/Solver Loop...")
result = challenger_solver_loop(
    problem="Write a function to sort a list",
    solver_fn=mock_solver,
    challenger_fn=mock_challenger,
    max_iterations=3,
    consensus_threshold=0.8
)

print(f"\nSuccess: {result.success}")
print(f"Iterations: {result.iterations}")
print(f"Final Solution: {result.final_solution}")
print(f"\nCritiques:")
for i, critique in enumerate(result.critiques):
    print(f"  {i+1}. Score: {critique.score}, Feedback: {critique.feedback}")

# Assertions
assert result.success == True
assert result.iterations == 2
assert "if lst else" in result.final_solution
print("\nVerification passed!")

Running Challenger/Solver Loop...

Success: True
Iterations: 2
Final Solution: def sort_list(lst): return sorted(lst) if lst else []

Critiques:
  1. Score: 0.5, Feedback: What about empty list?
  2. Score: 0.9, Feedback: Solution handles empty list correctly.

Verification passed!
